# 🏥 Clinical Decision Support (CDS) — Vector Embedding & ChromaDB Indexing

This notebook loads section-aware chunks from `data/processed_chunks.json`, computes dense biomedical embeddings using **`pritamdeka/S-PubMedBert-MS-MARCO`**, and indexes them into a persistent **ChromaDB** vector database with complete clinical metadata.

### **Key Highlights**:
- **Biomedical S-PubMedBERT Model**: Pre-trained on PubMed and fine-tuned for MS-MARCO clinical passage retrieval.
- **Section-Context Preservation**: Embeds `embedding_text` containing section headers for optimal semantic grounding.
- **ChromaDB Vector Store**: Persisted locally at `data/chroma_db` with cosine similarity indexing.
- **Metadata Schema**: Preserves Document Name, Section Number/Title, Page Number, Evidence Grade, and Target Population.

## Using `GH_TOKEN` for GitHub Operations

If your repository is private or you experience rate limiting, you can use a GitHub Personal Access Token (PAT) for authentication.

1.  **Create a PAT**: Go to GitHub > Settings > Developer settings > Personal access tokens > Tokens (classic) > Generate new token.
2.  **Add to Colab Secrets**: In Colab, go to the "🔑" icon in the left panel, click "Secrets", and add your PAT with the name `GH_TOKEN`.
3.  **Use in Clone Command**: You can then use it in the `git clone` command as shown below.

In [1]:
# Import the `userdata` module to securely access your GitHub Token
from google.colab import userdata
import os

# Get the GH_TOKEN from Colab Secrets (if available)
# userdata.get() returns None if the key is not found
GH_TOKEN = userdata.get('GH_TOKEN')

# Construct the repository URL, including the token if it exists
repo_url = "https://github.com/Abdelrahmann-Mostafa/Pyramind---Hackathon.git"
if GH_TOKEN is not None:
    # Insert the token into the URL for authentication
    repo_url = repo_url.replace("https://", f"https://oauth2:{GH_TOKEN}@")

# Clone the repository
!git clone {repo_url}

print("Repository cloned successfully!")

Cloning into 'Pyramind---Hackathon'...
remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 40 (delta 12), reused 26 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (40/40), 401.85 KiB | 2.02 MiB/s, done.
Resolving deltas: 100% (12/12), done.
Repository cloned successfully!


The previous cells indicate that `processed_chunks.json` is missing and suggest running `src/ingestion.py`. I will move the `data` and `src` directories from the cloned repository to the current working directory (`/content/`) so that the notebook can find these files. You may need to run `src/ingestion.py` if `processed_chunks.json` is still not found in `data` after this step.

In [2]:
# Move the 'data' and 'src' directories from the cloned repository to the current working directory

# Define the path to the cloned repository
cloned_repo_path = "./Pyramind---Hackathon"

# Move 'data' directory
if os.path.exists(os.path.join(cloned_repo_path, "data")):
    !mv {cloned_repo_path}/data .
    print("Moved 'data' directory.")
else:
    print("'data' directory not found in the cloned repository.")

# Move 'src' directory
if os.path.exists(os.path.join(cloned_repo_path, "src")):
    !mv {cloned_repo_path}/src .
    print("Moved 'src' directory.")
else:
    print("'src' directory not found in the cloned repository.")

# Clean up the cloned repository directory if it's empty or no longer needed
# You can uncomment the following lines if you want to remove the empty repo directory
# if not os.listdir(cloned_repo_path):
#     os.rmdir(cloned_repo_path)
#     print("Removed empty cloned repository directory.")

Moved 'data' directory.
Moved 'src' directory.


Now that the `data` and `src` directories have been moved, you should be able to proceed. You might need to re-run the cells that caused the `FileNotFoundError` (specifically Step 2) and potentially run `src/ingestion.py` if the `processed_chunks.json` file is generated by it.

In [3]:
# Step 1: Install dependencies (run this if executing on Google Colab)
!pip install sentence-transformers chromadb pydantic torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.1 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found

In [4]:
import os
import json
from pathlib import Path
import torch
import chromadb
from sentence_transformers import SentenceTransformer

# Determine project root path automatically
CURRENT_DIR = Path(os.getcwd())
if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

CHUNKS_FILE = PROJECT_ROOT / "data" / "processed_chunks.json"
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma_db"
COLLECTION_NAME = "clinical_guidelines"
MODEL_NAME = "pritamdeka/S-PubMedBert-MS-MARCO"

# Hardware Acceleration (CUDA GPU / CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[+] Project Root: {PROJECT_ROOT}")
print(f"[+] Chunks File:  {CHUNKS_FILE}")
print(f"[+] Chroma DB:    {CHROMA_DIR}")
print(f"[+] Using Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

[+] Project Root: /content
[+] Chunks File:  /content/data/processed_chunks.json
[+] Chroma DB:    /content/data/chroma_db
[+] Using Device: cuda (Tesla T4)


In [5]:
# Step 2: Load processed section-aware chunks
if not CHUNKS_FILE.exists():
    raise FileNotFoundError(f"Chunks file not found at {CHUNKS_FILE}. Please run src/ingestion.py first.")

with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"[+] Successfully loaded {len(chunks)} chunks from {CHUNKS_FILE.name}")
print("=" * 60)
print(f"Sample Chunk ID:       {chunks[0]['chunk_id']}")
print(f"Sample Document:       {chunks[0]['document_name']}")
print(f"Sample Section:        {chunks[0]['section_number']} - {chunks[0]['section_title']}")
print(f"Sample Evidence Grade: {chunks[0]['evidence_grade']}")
print(f"Sample Population:     {chunks[0]['target_population']}")
print("=" * 60)

[+] Successfully loaded 135 chunks from processed_chunks.json
Sample Chunk ID:       NICE_CG124_p01_c001
Sample Document:       NICE_CG124.pdf
Sample Section:        0.0 - Overview & Scope
Sample Evidence Grade: N/A
Sample Population:     Adults with acute hip fracture


In [7]:
# Install PyMuPDF (fitz) if not already installed
!pip install PyMuPDF

# Run the ingestion script to generate 'processed_chunks.json'
!python src/ingestion.py

print("Ingestion script executed. Attempting to re-run the chunk loading cell now.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 79.4 MB/s eta 0:00:00
CLINICAL GUIDELINE INGESTION & SECTION-AWARE CHUNKING PIPELINE

[+] Ingesting: NICE_CG124.pdf
    - Extracted 28 physical pages.
    - Generated 122 section-aware chunks.

[+] Ingesting: USPSTF_Osteoporosis.pdf
    - Extracted 4 physical pages.
    - Generated 13 section-aware chunks.

PIPELINE COMPLETE: Successfully processed and indexed 135 chunks.
Output saved to: /content/data/processed_chunks.json

--- SAMPLE CHUNKS PREVIEW ---
{
  "chunk_id": "NICE_CG124_p01_c001",
  "content": "Hip fracture: management\nClinical guideline\nPublished: 22 June 2011\nLast updated: 6 January 2023\nwww.nice.org.uk/guidance/cg124\n\u00a9 NICE 2026. All rights reserved. Subject to Notice of rights (https://www.nice.org.uk/terms-and-\nconditions#notice-of-rights).",
  "embedding_text": "[Section 0.0: Overview & Scope] Hip fracture: management\nClinical guideline\nPublished: 22 June 2011\nLast updated: 6 January 2023\nwww.nice

In [8]:
# Step 3: Load Domain-Adapted Biomedical Model & Generate Embeddings
print(f"[+] Loading '{MODEL_NAME}' on {device}...")
model = SentenceTransformer(MODEL_NAME, device=device)

# Use embedding_text (includes section headers) for optimal semantic retrieval
texts_to_embed = [chunk.get("embedding_text", chunk["content"]) for chunk in chunks]

print(f"[+] Generating dense embeddings for {len(texts_to_embed)} chunks...")
embeddings = model.encode(
    texts_to_embed,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True  # Cosine similarity via normalized inner product
)

print(f"\n[+] Embedding matrix shape: {embeddings.shape} (Chunks x Vector Dim)")

[+] Loading 'pritamdeka/S-PubMedBert-MS-MARCO' on cuda...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[+] Generating dense embeddings for 135 chunks...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]


[+] Embedding matrix shape: (135, 768) (Chunks x Vector Dim)


In [9]:
# Step 4: Index & Store in Persistent ChromaDB
os.makedirs(CHROMA_DIR, exist_ok=True)
client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Re-create clean collection with cosine distance metric
try:
    client.delete_collection(name=COLLECTION_NAME)
    print(f"[+] Reset existing collection '{COLLECTION_NAME}'")
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

# Format records for ChromaDB
ids = [chunk["chunk_id"] for chunk in chunks]
documents = [chunk["content"] for chunk in chunks]
metadatas = [
    {
        "document_name": str(chunk.get("document_name", "")),
        "section_number": str(chunk.get("section_number", "")),
        "section_title": str(chunk.get("section_title", "")),
        "page_number": int(chunk.get("page_number", 1)),
        "evidence_grade": str(chunk.get("evidence_grade", "N/A")),
        "target_population": str(chunk.get("target_population", "Adults")),
        "char_count": int(chunk.get("char_count", len(chunk["content"]))),
    }
    for chunk in chunks
]
embeddings_list = embeddings.tolist()

# Batch insertion
BATCH_SIZE = 64
for i in range(0, len(ids), BATCH_SIZE):
    end_i = min(i + BATCH_SIZE, len(ids))
    collection.add(
        ids=ids[i:end_i],
        embeddings=embeddings_list[i:end_i],
        documents=documents[i:end_i],
        metadatas=metadatas[i:end_i]
    )

print(f"[+] Indexing complete: {collection.count()} chunks stored in ChromaDB at '{CHROMA_DIR}'!")

[+] Indexing complete: 135 chunks stored in ChromaDB at '/content/data/chroma_db'!


## 🧪 Step 5: Verification & Semantic Retrieval Test

Let's test semantic similarity queries against our indexed guidelines to confirm retrieval precision and metadata surfacing.

In [10]:
def query_guidelines(query_str: str, top_k: int = 3):
    print("\n" + "=" * 80)
    print(f"CLINICAL QUERY: \"{query_str}\"")
    print("=" * 80)

    query_vec = model.encode([query_str], normalize_embeddings=True).tolist()

    results = collection.query(
        query_embeddings=query_vec,
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    for rank in range(top_k):
        c_id = results["ids"][0][rank]
        content = results["documents"][0][rank]
        meta = results["metadatas"][0][rank]
        dist = results["distances"][0][rank]
        sim = 1.0 - dist  # Cosine similarity score

        print(f"\n[Result #{rank+1}] Similarity: {sim:.4f} | Chunk: {c_id}")
        print(f"  - Document:          {meta['document_name']} (Page {meta['page_number']})")
        print(f"  - Section:           {meta['section_number']} — {meta['section_title']}")
        print(f"  - Evidence Grade:    {meta['evidence_grade']}")
        print(f"  - Target Population: {meta['target_population']}")
        print(f"  - Excerpt:           {content[:250].strip()}...")

# Test 1: Hip Fracture Analgesia (NICE CG124)
query_guidelines("What is the recommended analgesia regimen for hip fracture patients upon admission?", top_k=2)

# Test 2: Osteoporosis Screening in Older Women (USPSTF)
query_guidelines("What are the screening recommendations for osteoporosis in women aged 65 and older?", top_k=2)

# Test 3: Surgical Timing (NICE CG124)
query_guidelines("What is the recommended surgical timing for medically fit hip fracture patients?", top_k=2)


CLINICAL QUERY: "What is the recommended analgesia regimen for hip fracture patients upon admission?"

[Result #1] Similarity: 0.9523 | Chunk: NICE_CG124_p07_c020
  - Document:          NICE_CG124.pdf (Page 7)
  - Section:           1.3 — Analgesia
  - Evidence Grade:    N/A
  - Target Population: Adults with acute hip fracture
  - Excerpt:           [2011]
1.3.2
Offer immediate analgesia to people presenting at hospital with suspected hip
fracture, including people with cognitive impairment.

[2011]
1.3.3
Ensure analgesia is sufficient to allow movements necessary for investigations
(as indicate...

[Result #2] Similarity: 0.9519 | Chunk: NICE_CG124_p07_c019
  - Document:          NICE_CG124.pdf (Page 7)
  - Section:           1.3 — Analgesia
  - Evidence Grade:    N/A
  - Target Population: Adults with acute hip fracture
  - Excerpt:           1.3.1
Assess the person's pain:
• immediately upon presentation at hospital and
• within 30 minutes of administering initial analgesia and
• 